In [1]:
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
from pyproj import CRS, Transformer

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except Exception:
    widgets = None
    clear_output = None
    display = None

In [2]:
# Hardwritten values and paths
DEFAULT_INPUT_PATH = r""
DEFAULT_TARGET_CRS = "EPSG:3812"
DEFAULT_SAMPLE_ENABLED = True
DEFAULT_SAMPLE_EVERY_M = 0.005
DEFAULT_OUTPUT_FOLDER_NAME = "PLY"
OVERWRITE_OUTPUT = True

In [ ]:
# Run UI

if widgets is None:
    print("ipywidgets is not available.")
else:
    def collect_input_files(input_path: Path):
        if not input_path.exists():
            raise FileNotFoundError(f"Input path does not exist: {input_path}")
        if input_path.is_file():
            if input_path.suffix.lower() not in {".kmz", ".zip", ".kml"}:
                raise ValueError(f"Unsupported file type: {input_path.suffix}")
            return [input_path]
        return sorted(
            p for p in input_path.rglob("*")
            if p.is_file() and p.suffix.lower() in {".kmz", ".zip", ".kml"}
        )

    def safe_relative(path: Path, root: Path):
        try:
            return path.relative_to(root)
        except Exception:
            return Path(path.name)

    def extract_kml_from_archive(src_archive: Path, kml_dir: Path, root: Path):
        kml_dir.mkdir(parents=True, exist_ok=True)
        rel = safe_relative(src_archive, root)
        out_dir = kml_dir / rel.parent
        out_dir.mkdir(parents=True, exist_ok=True)
        out_kml = out_dir / f"{src_archive.stem}__template.kml"

        with zipfile.ZipFile(src_archive, "r") as zf:
            names = zf.namelist()
            candidates = [n for n in names if n.lower().endswith("template.kml")]
            if not candidates:
                kml_files = [n for n in names if n.lower().endswith(".kml")]
                if len(kml_files) == 1:
                    candidates = kml_files
            if not candidates:
                raise FileNotFoundError(f"No template.kml (or unique .kml) in: {src_archive}")
            out_kml.write_bytes(zf.read(candidates[0]))

        return out_kml

    def _find_float(el, xpath: str):
        node = el.find(xpath)
        if node is None or node.text is None:
            return None
        txt = node.text.strip()
        if not txt:
            return None
        try:
            return float(txt)
        except Exception:
            return None

    def parse_waypoints_in_order(kml_path: Path):
        root = ET.parse(kml_path).getroot()
        waypoints = []

        for i, pm in enumerate(root.findall(".//{*}Placemark")):
            node = pm.find(".//{*}Point/{*}coordinates")
            if node is None or not node.text:
                continue

            parts = [p.strip() for p in node.text.strip().split(",")]
            if len(parts) < 2:
                continue

            try:
                lon = float(parts[0])
                lat = float(parts[1])
            except Exception:
                continue

            coord_alt = None
            if len(parts) >= 3 and parts[2]:
                try:
                    coord_alt = float(parts[2])
                except Exception:
                    coord_alt = None

            # WPML priority: ellipsoidHeight, then height, then optional coordinate Z.
            h_ellipsoid = _find_float(pm, ".//{*}ellipsoidHeight")
            h_wpml = _find_float(pm, ".//{*}height")

            if h_ellipsoid is not None:
                h = h_ellipsoid
            elif h_wpml is not None:
                h = h_wpml
            elif coord_alt is not None:
                h = coord_alt
            else:
                raise ValueError(
                    f"Missing waypoint height at Placemark #{i} in {kml_path.name}. "
                    "Expected wpml:ellipsoidHeight or wpml:height (or coordinate Z)."
                )

            waypoints.append((lon, lat, h))

        return waypoints

    def transform_waypoints(waypoints_llh, target_crs):
        dst = CRS.from_user_input(target_crs)
        tr = Transformer.from_crs(CRS.from_epsg(4979), dst, always_xy=True)
        xyz = []
        for lon, lat, h in waypoints_llh:
            res = tr.transform(lon, lat, h)
            if len(res) >= 3:
                xyz.append((float(res[0]), float(res[1]), float(res[2])))
            else:
                xyz.append((float(res[0]), float(res[1]), float(h)))
        return xyz, dst

    def remove_consecutive_duplicates(points_xyz, eps=1e-9):
        if not points_xyz:
            return []
        out = [points_xyz[0]]
        for p in points_xyz[1:]:
            if np.linalg.norm(np.asarray(p, dtype=float) - np.asarray(out[-1], dtype=float)) > eps:
                out.append(p)
        return out

    def cumulative_distances(points_xyz):
        p = np.asarray(points_xyz, dtype=float)
        if len(p) == 0:
            return np.array([], dtype=float)
        if len(p) == 1:
            return np.array([0.0], dtype=float)
        seg = np.linalg.norm(np.diff(p, axis=0), axis=1)
        return np.concatenate([[0.0], np.cumsum(seg)])

    def sample_polyline(points_xyz, spacing_m, eps=1e-12):
        p = np.asarray(points_xyz, dtype=float)
        if len(p) == 0:
            return np.empty((0, 3), dtype=float), np.array([], dtype=float), np.array([], dtype=int)
        if len(p) == 1:
            return p.copy(), np.array([0.0], dtype=float), np.array([0], dtype=int)

        spacing_m = float(spacing_m)
        if spacing_m <= 0:
            raise ValueError("spacing_m must be > 0")

        out = [p[0].copy()]
        s_out = [0.0]
        main_vertex_idx = [0]

        total_s = 0.0
        carry = spacing_m  # distance from current segment start to next regular sample

        for i in range(len(p) - 1):
            a = p[i]
            b = p[i + 1]
            v = b - a
            L = float(np.linalg.norm(v))

            if L <= eps:
                # Degenerate segment
                if np.linalg.norm(out[-1] - b) > eps:
                    out.append(b.copy())
                    s_out.append(total_s)
                main_vertex_idx.append(len(out) - 1)
                continue

            u = v / L
            d = carry

            # Regular samples on this segment (strictly interior)
            while d < L - eps:
                q = a + d * u
                if np.linalg.norm(out[-1] - q) > eps:
                    out.append(q)
                    s_out.append(total_s + d)
                d += spacing_m

            # Always include the endpoint (main waypoint)
            if np.linalg.norm(out[-1] - b) > eps:
                out.append(b.copy())
                s_out.append(total_s + L)
            main_vertex_idx.append(len(out) - 1)

            # Carry remainder to next segment
            overshoot = d - L
            if overshoot <= eps:
                carry = spacing_m
            else:
                carry = spacing_m - overshoot
                if carry <= eps:
                    carry = spacing_m

            total_s += L

        return np.asarray(out), np.asarray(s_out), np.asarray(main_vertex_idx, dtype=int)

    def build_main_edges(main_vertex_idx):
        edges = []
        for i in range(len(main_vertex_idx) - 1):
            a = int(main_vertex_idx[i])
            b = int(main_vertex_idx[i + 1])
            if a != b:
                edges.append((a, b))
        return edges

    def write_ply_ascii_with_main_edges(out_ply: Path, xyz_points, edges, dst_crs: CRS, overwrite=True):
        out_ply.parent.mkdir(parents=True, exist_ok=True)
        if out_ply.exists() and not overwrite:
            return out_ply

        n_v = len(xyz_points)
        n_e = len(edges)

        with out_ply.open("w", encoding="utf-8", newline="\n") as f:
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write("comment Ordered flight-route samples\n")
            f.write(f"comment CRS: {dst_crs.to_string()}\n")
            f.write(f"element vertex {n_v}\n")
            f.write("property double x\n")
            f.write("property double y\n")
            f.write("property double z\n")
            f.write(f"element edge {n_e}\n")
            f.write("property int vertex1\n")
            f.write("property int vertex2\n")
            f.write("end_header\n")

            for x, y, z in xyz_points:
                f.write(f"{float(x):.12f} {float(y):.12f} {float(z):.12f}\n")
            for a, b in edges:
                f.write(f"{a} {b}\n")

        return out_ply
    def process_route(src_file: Path, root: Path, target_crs: str, sample_enabled: bool, spacing_m: float, out_folder_name: str, overwrite: bool):
        kml_dir = root / "KML"
        out_dir = root / out_folder_name

        if src_file.suffix.lower() in {".kmz", ".zip"}:
            kml_path = extract_kml_from_archive(src_file, kml_dir, root)
        else:
            kml_path = src_file

        # 1) Extract ordered waypoints
        waypoints_llh = parse_waypoints_in_order(kml_path)
        if len(waypoints_llh) < 2:
            raise ValueError("Need at least 2 waypoints")

        # 2) Transform to target CRS + datum
        xyz_main, dst_crs = transform_waypoints(waypoints_llh, target_crs)

        # 3) Connect consecutive non-identical waypoints
        xyz_main = remove_consecutive_duplicates(xyz_main)
        if len(xyz_main) < 2:
            raise ValueError("All consecutive waypoints are identical")

        if sample_enabled:
            # 4) Sample each X m along straight segment chain (polyline interpolation)
            xyz_sampled, _, main_vertex_idx = sample_polyline(xyz_main, spacing_m)
            xyz_out = [tuple(v.tolist()) for v in xyz_sampled]
        else:
            xyz_out = [tuple(v) for v in xyz_main]
            main_vertex_idx = np.arange(len(xyz_main), dtype=int)

        # Edges connect only original main waypoints
        edges = build_main_edges(main_vertex_idx)

        # 5) Ordered ASCII PLY output
        rel = safe_relative(src_file, root)
        out_ply = (out_dir / rel).with_suffix(".ply")
        write_ply_ascii_with_main_edges(out_ply, xyz_out, edges, dst_crs, overwrite=overwrite)

        return {
            "source": str(src_file),
            "kml": str(kml_path),
            "out_ply": str(out_ply),
            "n_waypoints": len(xyz_main),
            "n_sampled": len(xyz_out),
            "n_edges": len(edges),
            "status": "OK",
        }

    # UI
    path_w = widgets.Text(
        value=DEFAULT_INPUT_PATH,
        placeholder=r"C:\path\to\route.kmz or folder",
        description="Input:",
        layout=widgets.Layout(width="95%"),
    )
    crs_w = widgets.Text(
        value=DEFAULT_TARGET_CRS,
        description="Target CRS:",
        layout=widgets.Layout(width="280px"),
    )
    sample_enabled_w = widgets.Checkbox(value=bool(DEFAULT_SAMPLE_ENABLED), description="Sample")
    spacing_w = widgets.FloatText(
        value=float(DEFAULT_SAMPLE_EVERY_M),
        description="Step [m]:",
        layout=widgets.Layout(width="190px"),
    )
    out_name_w = widgets.Text(
        value=DEFAULT_OUTPUT_FOLDER_NAME,
        description="Out dir:",
        layout=widgets.Layout(width="220px"),
    )
    overwrite_w = widgets.Checkbox(value=bool(OVERWRITE_OUTPUT), description="Overwrite")

    run_btn = widgets.Button(description="Run", button_style="success")
    prog = widgets.IntProgress(value=0, min=0, max=1, description="Routes", bar_style="warning", layout=widgets.Layout(width="430px"))
    status_w = widgets.HTML(value="<span style='color:#666;'>Ready</span>")
    out = widgets.Output()

    def _set_status(msg, color="#666"):
        status_w.value = f"<span style='color:{color};'>{msg}</span>"

    def _run(_):
        with out:
            clear_output()

            raw_path = str(path_w.value).strip().strip('"').strip("'")
            if not raw_path:
                _set_status("Input path missing", "#b00020")
                print("Please set an input path")
                return

            spacing = None
            if bool(sample_enabled_w.value):
                try:
                    spacing = float(spacing_w.value)
                except Exception:
                    _set_status("Invalid spacing", "#b00020")
                    print("Invalid sample spacing")
                    return
                if spacing <= 0:
                    _set_status("Spacing must be > 0", "#b00020")
                    print("Sample spacing must be > 0")
                    return

            input_path = Path(raw_path)
            try:
                files = collect_input_files(input_path)
            except Exception as e:
                _set_status("Input error", "#b00020")
                print(f"ERROR: {e}")
                return

            if not files:
                _set_status("No routes found", "#b00020")
                print("No .kmz/.zip/.kml files found")
                return

            root = input_path if input_path.is_dir() else input_path.parent
            out_folder = str(out_name_w.value).strip() or "PLY"

            prog.max = len(files)
            prog.value = 0

            ok_count = 0
            for i, src in enumerate(files, start=1):
                _set_status(f"Processing {i}/{len(files)}: {src.name}", "#666")
                try:
                    r = process_route(
                        src_file=src,
                        root=root,
                        target_crs=str(crs_w.value).strip(),
                        sample_enabled=bool(sample_enabled_w.value),
                        spacing_m=(spacing if spacing is not None else 0.0),
                        out_folder_name=out_folder,
                        overwrite=bool(overwrite_w.value),
                    )
                    ok_count += 1
                    print(f"OK  {src.name} -> {Path(r['out_ply']).name} ({r['n_waypoints']} wp -> {r['n_sampled']} pts, {r['n_edges']} edges)")
                except Exception as e:
                    print(f"ERR {src.name} -> {e}")
                prog.value = i

            print()
            print(f"Done: {ok_count}/{len(files)} routes converted")
            print(f"Output root: {root / out_folder}")
            _set_status(f"Done: {ok_count}/{len(files)} routes", "#0a7f2e")

    run_btn.on_click(_run)

    display(widgets.VBox([
        path_w,
        widgets.HBox([crs_w, sample_enabled_w, spacing_w, out_name_w, overwrite_w]),
        widgets.HBox([run_btn, prog]),
        status_w,
        out,
    ]))